<a href="https://colab.research.google.com/github/OmarEl-Zayat/displaced-population-analysis/blob/main/notebooks/displaced_population_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step1:




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_excel('/content/4_6007878812891618215.xlsx')

> **Note:** This notebook was originally run in Google Colab, where the dataset was loaded from Colab's temporary session storage (`/content/...`). To reproduce locally, update the file path in the cell above to point to your own copy of the dataset.

In [ ]:
df.head(10)

In [ ]:
df.tail(10)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

Step2:

In [ ]:
df['Date']=pd.to_datetime(df['Date'])

In [ ]:
df['year']=df['Date'].dt.year
df['month']=df['Date'].dt.month
df['day']=df['Date'].dt.day

In [ ]:
df[['Date', 'year', 'month', 'day']].head()

In [ ]:
df.isnull().sum()

In [ ]:
Total_Cells=df.shape[0]*df.shape[1]
Total_Missing=df.isnull().sum().sum()
Missing_Percentage=(Total_Missing/Total_Cells)*100
print(Missing_Percentage)


In [ ]:
Missing_Cols=['Flat','Room', 'House', 'Tent']
df[Missing_Cols]=df[Missing_Cols].fillna(0)

In [ ]:
text_col=['Country','City']
for col in text_col:
  df[col]=df[col].astype(str).str.strip()

In [ ]:
Total_Cells=df.shape[0]*df.shape[1]
Total_Missing_After_Handling=df.isnull().sum().sum()
Complete_Percentage=((Total_Cells - Total_Missing_After_Handling)/Total_Cells)*100
print(Complete_Percentage)


Step3:

In [ ]:
df['Male_Percentage']=np.where(df['Members']>0,(df['Male']/df['Members']*100), 0)
df['Female_Percentage']=np.where(df['Members']>0,(df['Female']/df['Members']*100), 0)

In [ ]:
df['Avg_People_Per_House']=np.where(df['House']>0,(df['Members']/df['House']), 0)

In [ ]:
housing_type= ['Flat', 'Room', 'House', 'Tent']
df['Dominant_Housing']=df[housing_type].idxmax(axis=1)

In [ ]:
df[['Male_Percentage', 'Female_Percentage', 'Avg_People_Per_House', 'Dominant_Housing']].head()

Step4:

In [ ]:
country_summary=df.groupby('Country')[['Members', 'Male', 'Female']].sum()
country_summary

In [ ]:
country_year_summary=df.groupby(['Country', 'year'])[['Members']].sum()
country_year_summary

In [ ]:
country_year_summary=df.groupby(['Country', 'year'])[['Members']].sum().unstack()
country_year_summary

In [ ]:
house_pivot_table=pd.pivot_table(df, index='Country', columns='year', values='House', aggfunc='sum')
house_pivot_table

Step5:

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
features=['Houses', 'Members', 'Avg_People_Per_House']
m=MinMaxScaler()
s=StandardScaler()
df[[f'{col}_scaled' for col in features]]=m.fit_transform(df[features])
df[['Houses_scaled', 'Members_scaled', 'Avg_People_Per_House_scaled']].head()


In [ ]:
df['Members_zscore']=s.fit_transform(df[['Members']])
df[['Members', 'Members_zscore']].head()

In [ ]:
outliers=df[df['Members_zscore'].abs()>3]
outliers.head()

Step6:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
country_members=df.groupby('Country')['Members'].sum()
plt.bar(country_members.index, country_members.values)
plt.xlabel('Country')
plt.ylabel('Total Members')
plt.title('Total Members by Country')
plt.show()


In [ ]:
year_country=df.groupby(['year', 'Country'])['Members'].sum().reset_index()
sns.lineplot(x='year', y='Members', hue='Country', data=year_country)
plt.xlabel('Year')
plt.ylabel('Total Members')
plt.title('Total Members by Year and Country')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.kdeplot(data=df, x='Male_Percentage', label='Male Percentage', fill=True, alpha=0.6, linewidth=2)
sns.kdeplot(data=df, x='Female_Percentage', label='Female Percentage', fill=True, alpha=0.6, linewidth=2)
plt.xlabel('Percentage')
plt.ylabel('Density')
plt.title('Distribution of Male and Female Percentages')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
sns.regplot(data=df, x='Houses', y='Members')
plt.xlabel('Houses')
plt.ylabel('Members')
plt.title('Relationship Between Houses & Members Relationship')
plt.show()

In [ ]:
correlation=df.corr(numeric_only=True)
plt.figure(figsize=(12, 8))
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()